In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaLLM
import warnings
warnings.filterwarnings('ignore')

In [2]:
loader = PyPDFLoader("policy.pdf")
docs = loader.load()


# We are loading the company policy data pdf file

In [3]:
# Split
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

#| Parameter          | Meaning                                  |
#| ------------------ | ---------------------------------------- |
#| `chunk_size=500`   | each chunk contains ~500 characters      |
#| `chunk_overlap=50` | 50 characters repeated in the next chunk |

#Chunk 1 → characters 1–500
#Chunk 2 → characters 450–950
#Chunk 3 → characters 900–1400


# We are dividing/splitting the data in to chunks



In [4]:
# Embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# Vector DB
vectorstore = FAISS.from_documents(chunks, embeddings)



# We are converting the chunks in to Word Embeddings.

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# Retriever
retriever = vectorstore.as_retriever()

In [6]:
# LLM
llm = OllamaLLM(model="llama3")


# 4 to 6 gb space. 

In [32]:
query = str(input())

 How many leaves does an employee get?


In [33]:
docs = retriever.invoke(query)

In [34]:
context = "\n".join([d.page_content for d in docs])

In [35]:
prompt = f"""
Answer the question using the context.

Context:
{context}

Question:
{query}

If the answer is not in the context, say "I don't know".

Answer:
"""

In [36]:
response = llm.invoke(prompt)

In [37]:
print(response)

An employee gets 20 days of paid annual leave per year. Additionally, they also receive 10 days of sick leave annually and 5 days of casual leave per year for personal matters.
